# Explore the Norwegian Continental Shelf with KGLite Visual

This notebook builds the public SODIR knowledge graph with `kglite-datasets`, opens the saved `.kgl` file in KGLite Visual, and follows a bounded geological journey through fields, discoveries, wells, formation tops, physical evidence, and monthly production.

Every graph query has an explicit limit. The app remains open after **Run All** so you can continue exploring. The final cleanup cells are optional.

**Source meaning.** Production values are monthly aggregates from the [SODIR field production table](https://factpages.sodir.no/en/field/TableView/Production/Saleable/TotalNcsMonth). Formation-top depth follows the [SODIR wellbore attributes](https://factpages.sodir.no/en/wellbore/PageView/Exploration/All/6374) and is shown as measured depth in metres from Kelly bushing (MD m RKB). This notebook does not infer daily telemetry, well trajectories, or reservoir connectivity.

## One-time environment setup

You need Git, Python 3.12 (the tested version), a Rust toolchain with Cargo, and Node.js with npm. Create an isolated environment first so the install does not depend on—or try to modify—your system Python. On Windows PowerShell, create it with `py -3.12 -m venv .venv` and activate it with `.\.venv\Scripts\Activate.ps1` instead of the first two commands below.

The workspace used here is newer than the published `kglite-visual 0.1.7` wheel. Build the reviewed preview into the notebook's environment before running it:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
git clone https://github.com/kkollsga/kglite-visual.git
cd kglite-visual
git checkout 61e0534a89057535ba5a638dc9a83e2e0281cd78
npm --prefix frontend ci
npm --prefix frontend run build
python -m pip install .
python -m pip install 'kglite==0.17.1' matplotlib jupyter
python -m pip install 'kglite-datasets @ git+https://github.com/kkollsga/kglite-datasets.git@4db84882a853060f61f0b27f9665a145b0b74bd3'
python -m ipykernel install --user --name kglite-sodir --display-name 'KGLite SODIR'
cd ..
jupyter lab
```

Download `sodir-geologist.ipynb` from the guide into the directory where you created `.venv`, then launch `jupyter lab` from that directory as shown above. The pinned source revision supplies the preview viewer; it predates this notebook. Open the downloaded notebook and select the **KGLite SODIR** kernel.

The source cache can be hundreds of megabytes; the validated 2026-09-08 graph was about 115 MB, and upstream refreshes can change its size and counts. Set `SODIR_PROJECT_DIR` before starting Jupyter to put them somewhere else. The default is `.kglite-sodir-notebook` beside this notebook. Cached source files are reused according to `kglite-datasets` cooldowns. The generated graph is reused only when its build record names the pinned datasets revision and its required discovery-play and volume capabilities pass a live query; otherwise the notebook rebuilds it. `SODIR_FORCE_REBUILD=1` still forces a rebuild.

In [ ]:
from __future__ import annotations

import calendar
import gc
import http.client
import html as html_lib
import importlib.metadata
import json
import math
import os
from pathlib import Path
from urllib.parse import urljoin

from IPython.display import HTML, Image, display
import matplotlib.pyplot as plt
import kglite
import kglite_visual as kv

KGLITE_VERSION = "0.17.1"
DATASETS_VERSION = "0.1.16"
DATASETS_REVISION = "4db84882a853060f61f0b27f9665a145b0b74bd3"
VIEWER_REVISION = "61e0534a89057535ba5a638dc9a83e2e0281cd78"
PROTOCOL_VERSION = 10

assert importlib.metadata.version("kglite") == KGLITE_VERSION
assert importlib.metadata.version("kglite-datasets") == DATASETS_VERSION

PROJECT_DIR = Path(os.environ.get("SODIR_PROJECT_DIR", ".kglite-sodir-notebook")).expanduser().resolve()
SOURCE_CACHE = PROJECT_DIR / "source-cache"
GRAPH_DIR = PROJECT_DIR / "graph"
EXPORT_DIR = PROJECT_DIR / "exports"
FIGURE_DIR = PROJECT_DIR / "figures"
GRAPH_PATH = GRAPH_DIR / "sodir.kgl"
PRODUCTION_PATH = GRAPH_DIR / "production-2024.json"
GRAPH_BUILD_PATH = GRAPH_DIR / "sodir-build.json"
CREAMING_GRAPH = os.environ.get("SODIR_CREAMING_GRAPH")
CREAMING_GRAPH_PATH = (Path(CREAMING_GRAPH).expanduser().resolve()
                       if CREAMING_GRAPH else None)
for directory in (SOURCE_CACHE, GRAPH_DIR, EXPORT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Project: {PROJECT_DIR.name}/")


## Build once, reuse deliberately

The loader owns `source-cache/`. A normal rerun reuses both that cache and the portable graph. `SODIR_FORCE_REBUILD=1` asks the loader to reconsider its cached inputs under its 14-day index and 30-day dataset cooldowns. While the builder graph is open, the next cell uses type-aware Cypher `ts_series` expressions to extract only three fields × two channels × twelve months into a small provenance sidecar. It then releases the builder before the file-backed viewer starts. KGLite 0.17.1 records a false static schema warning for these typed time-series pseudo-properties even though it returns their values. This cell temporarily silences only the warning announcement, inspects the structured warnings, rejects anything unexpected, and retains the exact length, date, unit, and value checks.

In [ ]:
FIELDS = ["JOHAN SVERDRUP", "TROLL", "EKOFISK"]
PRODUCTION_YEAR = 2024
assert 2 <= len(FIELDS) <= 3
assert 1960 <= PRODUCTION_YEAR <= 2100


def serial_value(value):
    if value is None:
        return None
    value = float(value)
    return value if math.isfinite(value) else None


def extract_production(graph):
    config = graph.timeseries_config("ProductionProfile")
    assert config["resolution"] == "month"
    assert config["units"]["prd_oil_net"] == "MillSm3"
    assert config["units"]["prd_gas_net"] == "BillSm3"
    production = {}
    year = str(PRODUCTION_YEAR)
    query = (
        "MATCH (p:ProductionProfile)-[:OF_FIELD]->(f:Field) "
        "WHERE f.title = $field "
        "RETURN ts_series(p.prd_oil_net, '{year}', '{year}') AS oil, "
        "ts_series(p.prd_gas_net, '{year}', '{year}') AS gas LIMIT 1"
    ).format(year=year)
    for field in FIELDS:
        warning_policy = kglite.get_query_warning_policy()
        kglite.set_query_warning_policy("silent")
        try:
            result = graph.cypher(query, params={"field": field})
        finally:
            kglite.set_query_warning_policy(warning_policy)
        unexpected = [message for message in result.warnings
                      if not ("RETURN projects property" in message
                              and "ProductionProfile node has" in message
                              and ("prd_oil_net" in message or "prd_gas_net" in message))]
        if unexpected:
            raise RuntimeError(f"unexpected production-query warning: {unexpected}")
        row = result.to_dicts()[0]
        def by_month(items):
            values = {}
            for item in items:
                month = str(item["time"])[:7]
                if not month.startswith(f"{PRODUCTION_YEAR}-") or month in values:
                    raise ValueError(f"unexpected or duplicate production month: {month}")
                values[month] = serial_value(item["value"])
            return values
        oil = by_month(row["oil"])
        gas = by_month(row["gas"])
        months = [f"{PRODUCTION_YEAR}-{month:02d}" for month in range(1, 13)]
        production[field] = [
            {"month": month, "oil_million_sm3": oil.get(month),
             "gas_billion_sm3": gas.get(month)}
            for month in months
        ]
    return {
        "provenance": {"kglite": KGLITE_VERSION, "kglite_datasets": DATASETS_VERSION,
                       "kglite_datasets_revision": DATASETS_REVISION,
                       "year": PRODUCTION_YEAR, "fields": FIELDS,
                       "source": "ProductionProfile packed monthly time series",
                       "resolution": config["resolution"],
                       "oil_unit": config["units"]["prd_oil_net"],
                       "gas_unit": config["units"]["prd_gas_net"]},
        "series": production,
    }

REQUIRED_GRAPH_CAPABILITIES = {
    "assigned_discoveries": "MATCH (d:Discovery)-[:IN_PLAY]->(:Play) RETURN count(DISTINCT d) AS count LIMIT 1",
    "candidate_assignments": "MATCH (d:Discovery)-[:CANDIDATE_PLAY]->(:Play) RETURN count(d) AS count LIMIT 1",
    "discovery_volumes": "MATCH (v:DiscoveryVolume)-[:OF_DISCOVERY]->(:Discovery) RETURN count(v) AS count LIMIT 1",
    "field_examples": "MATCH (f:Field)-[:IN_PLAY]->(:Play) RETURN count(f) AS count LIMIT 1",
}


def graph_capabilities(graph):
    return {name: int(graph.cypher(query).to_dicts()[0]["count"])
            for name, query in REQUIRED_GRAPH_CAPABILITIES.items()}


expected_build = {
    "kglite": KGLITE_VERSION,
    "kglite_datasets": DATASETS_VERSION,
    "kglite_datasets_revision": DATASETS_REVISION,
    "default_discovery_play_enhancement": True,
}
def graph_cache_is_reusable(build_record, expected, capabilities):
    return (isinstance(build_record, dict)
            and all(build_record.get(key) == value for key, value in expected.items())
            and isinstance(capabilities, dict)
            and set(capabilities) == set(REQUIRED_GRAPH_CAPABILITIES)
            and build_record.get("capabilities") == capabilities
            and all(isinstance(count, int) and count > 0
                    for count in capabilities.values()))


try:
    cached_build = json.loads(GRAPH_BUILD_PATH.read_text())
except (FileNotFoundError, json.JSONDecodeError):
    cached_build = None
force_rebuild = os.environ.get("SODIR_FORCE_REBUILD") == "1"
build_metadata_matches = (isinstance(cached_build, dict)
                          and all(cached_build.get(key) == value
                                  for key, value in expected_build.items()))
needs_graph = force_rebuild or not GRAPH_PATH.is_file() or not build_metadata_matches
if not needs_graph:
    cached_graph = kglite.load(str(GRAPH_PATH))
    try:
        cached_capabilities = graph_capabilities(cached_graph)
    finally:
        del cached_graph
    needs_graph = not graph_cache_is_reusable(
        cached_build, expected_build, cached_capabilities)

expected_series = {"kglite": KGLITE_VERSION, "kglite_datasets": DATASETS_VERSION,
                   "kglite_datasets_revision": DATASETS_REVISION,
                   "year": PRODUCTION_YEAR, "fields": FIELDS,
                   "source": "ProductionProfile packed monthly time series",
                   "resolution": "month", "oil_unit": "MillSm3", "gas_unit": "BillSm3"}
try:
    cached_production = json.loads(PRODUCTION_PATH.read_text())
except (FileNotFoundError, json.JSONDecodeError):
    cached_production = None
needs_series = (force_rebuild or not isinstance(cached_production, dict)
                or any(cached_production.get("provenance", {}).get(key) != value
                       for key, value in expected_series.items()))
if needs_graph:
    from kglite_datasets import sodir
    builder_graph = sodir.open(
        str(SOURCE_CACHE), storage="memory", index_cooldown_days=14,
        dataset_cooldown_days=30, use_complement=True, workers=10,
        force_rebuild=force_rebuild, verbose=True,
    )
    capabilities = graph_capabilities(builder_graph)
    if any(count <= 0 for count in capabilities.values()):
        raise RuntimeError(f"SODIR enhancement capabilities are incomplete: {capabilities}")
    production = extract_production(builder_graph)
    temporary_graph = GRAPH_PATH.with_suffix(".tmp.kgl")
    builder_graph.save(str(temporary_graph))
    os.replace(temporary_graph, GRAPH_PATH)
    build_record = {**expected_build, "capabilities": capabilities}
    temporary_build = GRAPH_BUILD_PATH.with_suffix(".tmp.json")
    temporary_build.write_text(json.dumps(build_record, indent=2) + "\n")
    os.replace(temporary_build, GRAPH_BUILD_PATH)
    del builder_graph
elif needs_series:
    builder_graph = kglite.load(str(GRAPH_PATH))
    production = extract_production(builder_graph)
    del builder_graph
else:
    production = cached_production

if needs_series or needs_graph:
    temporary_json = PRODUCTION_PATH.with_suffix(".tmp.json")
    temporary_json.write_text(json.dumps(production, indent=2) + "\n")
    os.replace(temporary_json, PRODUCTION_PATH)
gc.collect()

assert production["provenance"] == {
    "kglite": KGLITE_VERSION, "kglite_datasets": DATASETS_VERSION,
    "kglite_datasets_revision": DATASETS_REVISION,
    "year": PRODUCTION_YEAR, "fields": FIELDS,
    "source": "ProductionProfile packed monthly time series",
    "resolution": "month",
    "oil_unit": "MillSm3", "gas_unit": "BillSm3",
}
assert GRAPH_PATH.is_file() and GRAPH_PATH.stat().st_size > 0
print(f"Graph ready: {GRAPH_PATH.name} ({GRAPH_PATH.stat().st_size:,} bytes)")


## Start the file-backed workspace

Rerunning this cell closes only the viewer handle created by this notebook. The embedded app appears below in a local notebook; the link opens it in a separate tab so you can watch later cells update the same shared view. Remote kernels use `jupyter-server-proxy` when available or show an SSH forwarding hint.

In [ ]:
def request(method, route, body=None, *, raw=False, view=None):
    active_view = view or _sodir_view
    connection = http.client.HTTPConnection("127.0.0.1", active_view.port, timeout=45)
    try:
        payload = None if body is None else json.dumps(body)
        headers = {} if body is None else {"content-type": "application/json"}
        connection.request(method, route, payload, headers)
        response = connection.getresponse()
        data = response.read()
        if response.status >= 400:
            raise RuntimeError(f"{method} {route} -> {response.status}: {data[:500].decode(errors='replace')}")
        return (data, dict(response.getheaders())) if raw else json.loads(data)
    finally:
        connection.close()

previous_view = globals().get("_sodir_view")
if previous_view is not None and not previous_view.closed:
    previous_view.close()
os.environ["KGLITE_VISUAL_CONFIG_DIR"] = str(PROJECT_DIR / "saved-views")
_sodir_view = kv.show(str(GRAPH_PATH), open_browser=False, max_load_mb=1024,
                      name="SODIR NCS exploration", height=680)
try:
    session = request("GET", "/api/session")
    assert session["protocol_version"] == PROTOCOL_VERSION
    assert session["stats"]["node_count"] > 0 and session["stats"]["edge_count"] > 0
    print(f"Loaded {session['stats']['node_count']:,} nodes / {session['stats']['edge_count']:,} relations")
    request("POST", "/api/validate", {"query": "MATCH (f:Field) RETURN f LIMIT 1"})
except Exception:
    _sodir_view.close()
    raise RuntimeError(
        "This kernel does not have the reviewed Visual preview. Repeat the one-time setup above."
    )
display(HTML(f'<p><a href="{_sodir_view.url}" target="_blank"><b>Open the live SODIR workspace in a new tab</b></a></p>'))
display(_sodir_view)


In [ ]:
def require_complete(result, label):
    meta = result.get("meta", result)
    for key in ("bound", "link_bound"):
        bound = meta.get(key)
        if bound and bound["truncated"]:
            raise RuntimeError(
                f"{label} was truncated: {bound['returned']} of {bound['total']}. "
                "Narrow the query before interpreting it."
            )
    return result


def table_rows(query, params=None, *, label, view=None):
    result = request("POST", "/api/cypher", {
        "query": query, "params": params or {}, "limit": 100, "as_graph": False,
    }, view=view)
    require_complete(result, label)
    rows = [dict(zip(result["columns"], values)) for values in zip(*result["data"])]
    print(f"{label}: {result['bound']['returned']} of {result['bound']['total']} rows")
    return rows


def display_table(rows, title):
    if not rows:
        display(HTML(f"<p><b>{html_lib.escape(title)}:</b> no rows</p>"))
        return
    columns = list(rows[0])
    def cell(value):
        if value is None:
            return "<i>missing</i>"
        if isinstance(value, float):
            value = f"{value:,.2f}"
        return html_lib.escape(str(value))
    head = "".join(f"<th>{html_lib.escape(column)}</th>" for column in columns)
    body = "".join("<tr>" + "".join(f"<td>{cell(row.get(column))}</td>" for column in columns) + "</tr>" for row in rows)
    display(HTML(f"<h4>{html_lib.escape(title)}</h4><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))


def capture_image(path, *, kernel="auto", width=1200, height=800):
    current = request("GET", "/api/view-state")
    settings = {"scope": "visible", "format": "png", "expected": current["stamp"],
                "subset_revision": current["subset_revision"], "kernel": kernel,
                "width": width, "height": height, "seed": 7, "theme": "light"}
    preview = request("POST", "/api/render/preview", settings)
    payload, _ = request("POST", "/api/render/download",
                         {**settings, "preview_digest": preview["preview"]["preview_digest"]},
                         raw=True)
    path.write_bytes(payload)
    print(f"Rendered {preview['preview']['nodes']} nodes / {preview['preview']['edges']} relations: {path.name}")
    display(Image(filename=str(path)))
    return preview


def show_graph(query, params=None, *, label, kernel=None, color_by=None, reset=True):
    if reset:
        request("POST", "/api/reset", {})
    result = request("POST", "/api/cypher", {
        "query": query, "params": params or {}, "limit": 100, "as_graph": True,
    })
    require_complete(result, label)
    if kernel:
        request("POST", "/api/layout", {"kernel": kernel})
    if color_by:
        current = request("GET", "/api/view-state")
        request("POST", "/api/appearance", {
            "color_by": color_by, "size_by": None, "expected": current["stamp"],
        })
    node_bound = result["meta"]["bound"]
    link_bound = result["meta"]["link_bound"]
    print(f"Explore — {label}: {node_bound['returned']} nodes, {link_bound['returned']} relations")
    return result


## 1. Where are recently discovered producing fields?

**Question.** How widely are the twelve most recently discovered coordinate-bearing producing fields distributed across the NCS?

The query loads twelve distinct fields whose source records carry geometry and switches to the geographic layout. Open **Data**, add `fldHcType`, `fldDiscoveryYear`, and `wkt_geometry`, then use **Appearance → Color by → fldHcType**. Read each plotted point as a representative location derived from source geometry; it is not a field-outline or reservoir-extent map.

In [ ]:
MAP_QUERY = """MATCH (f:Field)
WHERE f.fldCurrentActivitySatus = 'Producing' AND f.wkt_geometry IS NOT NULL
RETURN f
ORDER BY f.fldDiscoveryYear DESC
LIMIT 12"""
map_view = show_graph(MAP_QUERY, label="12 producing fields", kernel="geo", color_by="fldHcType")
map_render = capture_image(FIGURE_DIR / "producing-fields-map.png", kernel="geo")


## 2. How did early Johan Sverdrup appraisal progress?

**Question.** What chronology is visible in the first twelve completed wildcat and appraisal wellbores linked through the Johan Sverdrup discovery?

The count and ordered table report current source data. The view shows at most the earliest twelve linked wellbores, so it is a bounded early chronology rather than the full drilling history. The picture shows recorded graph relationships, not well trajectories. In **Data**, inspect purpose, content, measured depth, and final vertical depth; select a well to see its path through the discovery to the field.

In [ ]:
appraisal_count = table_rows(
    "MATCH (f:Field)<-[:IN_FIELD]-(d:Discovery)<-[:IN_DISCOVERY]-(w:Wellbore) "
    "WHERE f.title = $field AND w.wlbPurpose IN ['WILDCAT', 'APPRAISAL'] "
    "RETURN count(*) AS linked_wells LIMIT 1",
    {"field": "JOHAN SVERDRUP"}, label="Linked wildcat and appraisal count")
linked_well_count = appraisal_count[0]["linked_wells"]
assert linked_well_count > 0
print(f"The next view is the earliest {min(12, linked_well_count)} of {linked_well_count} linked wellbores.")

FIELD_CONTEXT_QUERY = """MATCH (f:Field)<-[rf:IN_FIELD]-(d:Discovery)<-[rd:IN_DISCOVERY]-(w:Wellbore)
WHERE f.title = $field AND w.wlbPurpose IN ['WILDCAT', 'APPRAISAL']
RETURN f,rf,d,rd,w
ORDER BY w.wlbCompletionDate
LIMIT 12"""
field_view = show_graph(FIELD_CONTEXT_QUERY, {"field": "JOHAN SVERDRUP"},
                        label="Johan Sverdrup discovery wells", kernel="radial", color_by="type")
appraisal_rows = table_rows(
    "MATCH (f:Field)<-[:IN_FIELD]-(d:Discovery)<-[:IN_DISCOVERY]-(w:Wellbore) "
    "WHERE f.title = $field AND w.wlbPurpose IN ['WILDCAT', 'APPRAISAL'] "
    "RETURN w.title AS wellbore, w.wlbPurpose AS purpose, "
    "w.wlbCompletionDate AS completed, w.wlbContent AS content "
    "ORDER BY completed LIMIT 12",
    {"field": "JOHAN SVERDRUP"}, label="Early appraisal chronology")
display_table(appraisal_rows, "Early appraisal chronology")
if appraisal_rows:
    print(f"Current bounded chronology: {appraisal_rows[0]['wellbore']} completed {appraisal_rows[0]['completed']} "
          f"through {appraisal_rows[-1]['wellbore']} completed {appraisal_rows[-1]['completed']}.")


## 3. What depth context is recorded for well 16/2-6?

**Question.** Which formation-level tops were reported along the discovery well, and at what measured depths?

The table is ordered by `lsuTopDepth`, measured as **MD m RKB**. It is not TVD and does not demonstrate reservoir connectivity. `lsuBottomDepth` is deliberately excluded because it is unavailable in the current source snapshot. Read the aliased depth in the executed **Query results · source** table below; it is an edge property and is not a node field. Select EKOFISK FM or TOR FM there, then use **Show selection in Explore** to relate that record back to the well.

In [ ]:
FORMATION_GRAPH_QUERY = """MATCH (w:Wellbore)-[top:HAS_FORMATION_TOP]->(s:Stratigraphy)
WHERE w.title = $well AND s.lsuLevel = 'FORMATION'
RETURN w,top,s
ORDER BY top.lsuTopDepth
LIMIT 20"""
formation_view = show_graph(FORMATION_GRAPH_QUERY, {"well": "16/2-6"},
                            label="16/2-6 formation tops", kernel="radial", color_by="type")
formation_rows = table_rows(
    "MATCH (w:Wellbore)-[top:HAS_FORMATION_TOP]->(s:Stratigraphy) "
    "WHERE w.title = $well AND s.lsuLevel = 'FORMATION' "
    "RETURN s.title AS formation, top.lsuTopDepth AS top_md_m_rkb "
    "ORDER BY top_md_m_rkb LIMIT 20",
    {"well": "16/2-6"}, label="Formation tops")
display_table(formation_rows, "Formation tops — MD m RKB")


### Add core and DST intervals

**Question.** Which core and drill-stem-test source records are linked to 16/2-6?

This bounded view contains three core records and one DST record. The figure places their numeric source depths beside three nearby formation-top markers. SODIR documents the formation tops as MD m RKB and the DST depths in metres; the core source says metres but does not explicitly establish the same datum. Treat overlap as a prompt for source review, not a formation assignment or connectivity claim. The reported DST oil rate belongs to that test record; it is not field production.

In [ ]:
COMBINED_EVIDENCE_QUERY = """MATCH (w:Wellbore)-[r]-(e)
WHERE w.title = $well
  AND ((type(r) = 'HAS_FORMATION_TOP' AND e.type = 'Stratigraphy' AND e.lsuLevel = 'FORMATION')
    OR (type(r) = 'OF_WELLBORE' AND e.type IN ['WellboreCore', 'WellboreDST']))
RETURN w,r,e,coalesce(r.lsuTopDepth,e.wlbCoreIntervalTop,e.wlbDstFromDepth) AS top_md
ORDER BY top_md
LIMIT 40"""
evidence_view = show_graph(COMBINED_EVIDENCE_QUERY, {"well": "16/2-6"},
                           label="16/2-6 tops, cores, and DST", kernel="radial",
                           color_by="type")
core_rows = table_rows(
    "MATCH (c:WellboreCore)-[:OF_WELLBORE]->(w:Wellbore) WHERE w.title = $well "
    "RETURN c.wlbCoreNumber AS core, c.wlbCoreIntervalTop AS top_md, "
    "c.wlbCoreIntervalBottom AS bottom_md, c.wlbCoreIntervalUom AS unit, "
    "c.wlbTotalCoreLength AS recovered_length ORDER BY top_md LIMIT 12",
    {"well": "16/2-6"}, label="Core records")
dst_rows = table_rows(
    "MATCH (t:WellboreDST)-[:OF_WELLBORE]->(w:Wellbore) WHERE w.title = $well "
    "RETURN t.wlbDstTestNumber AS test, t.wlbDstFromDepth AS from_md_m, "
    "t.wlbDstToDepth AS to_md_m, t.wlbDstOilProd AS oil_sm3_day, "
    "t.wlbDstGasProd AS gas_sm3_day ORDER BY from_md_m LIMIT 12",
    {"well": "16/2-6"}, label="DST records")
display_table(core_rows, "Core intervals")
display_table(dst_rows, "Drill-stem test")
selected_tops = {"INTRA DRAUPNE FM SS", "DRAUPNE FM", "SKAGERRAK FM"}
fig, axis = plt.subplots(figsize=(9, 6))
for row in formation_rows:
    if row["formation"] in selected_tops:
        depth = row["top_md_m_rkb"]
        axis.scatter(0, depth, s=55)
        axis.annotate(f"{row['formation']}  {depth:g}", (0, depth), xytext=(8, 0),
                      textcoords="offset points", va="center")
for row in dst_rows:
    axis.vlines(1, row["from_md_m"], row["to_md_m"], linewidth=10, color="#d97706")
    axis.annotate(f"DST {row['from_md_m']:g}–{row['to_md_m']:g}",
                  (1, (row["from_md_m"] + row["to_md_m"]) / 2), xytext=(8, 0),
                  textcoords="offset points", va="center")
for row in core_rows:
    axis.vlines(2, row["top_md"], row["bottom_md"], linewidth=10, color="#059669")
    axis.annotate(f"Core {row['core']}  {row['top_md']:g}–{row['bottom_md']:g}",
                  (2, (row["top_md"] + row["bottom_md"]) / 2), xytext=(8, 0),
                  textcoords="offset points", va="center")
axis.set_xticks([0, 1, 2], ["Formation tops\nMD m RKB", "DST interval\nMD m", "Core intervals\nmetres; datum unspecified"])
axis.set_ylabel("Reported source depth (m)")
axis.set_title("16/2-6 — numeric depth context (no formation assignment)")
axis.set_xlim(-0.4, 2.8)
axis.invert_yaxis()
axis.grid(axis="y", alpha=.25)
fig.tight_layout()
depth_chart_path = FIGURE_DIR / "depth-context-16-2-6.png"
fig.savefig(depth_chart_path, dpi=150)
plt.show()


## 4. How are fields connected to monthly production?

**Question.** Which production profiles are attached to the three comparison fields?

The graph stores monthly values as packed time series on `ProductionProfile`, not as month nodes. The app can show the fields and profile relationships; the next cells read only the small 2024 sidecar extracted through KGLite's type-aware Cypher `ts_series` expression.

In [ ]:
PROFILE_QUERY = """MATCH (p:ProductionProfile)-[r:OF_FIELD]->(f:Field)
WHERE f.title IN $fields
RETURN p,r,f
LIMIT 12"""
profile_view = show_graph(PROFILE_QUERY, {"fields": FIELDS},
                         label=f"{len(FIELDS)} fields and production profiles", kernel="islands", color_by="type")


### Compare the complete common 2024 window

A monthly volume divided by that month's actual calendar days is labelled **monthly-average calendar-day rate**. February 2024 therefore uses 29 days. Missing source values stay missing. These curves are derived from monthly aggregates and are not day-by-day telemetry.

In [ ]:
def monthly_average_daily_rate(value, year, month, scale):
    if value is None:
        return None
    value = float(value)
    if not math.isfinite(value):
        return None
    return (value * scale) / calendar.monthrange(year, month)[1]


def rate_series(field, channel, scale):
    key = f"{channel}_{'million_sm3' if channel == 'oil' else 'billion_sm3'}"
    values = []
    for row in production["series"][field]:
        year, month = map(int, row["month"].split("-"))
        values.append(monthly_average_daily_rate(row[key], year, month, scale))
    return values

months = [row["month"] for row in production["series"]["JOHAN SVERDRUP"]]
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for field in FIELDS:
    axes[0].plot(months, rate_series(field, "oil", 1_000_000), marker="o", label=field)
    axes[1].plot(months, rate_series(field, "gas", 1_000_000_000), marker="o", label=field)
axes[0].set_ylabel("Oil (Sm³/day)\nmonthly average")
axes[1].set_ylabel("Gas (Sm³/day)\nmonthly average")
axes[1].set_xlabel(f"{PRODUCTION_YEAR} production month")
for axis in axes:
    axis.grid(alpha=.25)
    axis.legend(ncol=2)
axes[1].tick_params(axis="x", rotation=45)
fig.suptitle(f"{len(FIELDS)} NCS fields — monthly-average calendar-day production rates")
fig.tight_layout()
chart_path = FIGURE_DIR / f"production-{PRODUCTION_YEAR}.png"
fig.savefig(chart_path, dpi=150)
plt.show()

assert monthly_average_daily_rate(1, 2024, 2, 1_000_000) == 1_000_000 / 29
assert monthly_average_daily_rate(None, 2024, 2, 1_000_000) is None
johan_oil_total = sum(row["oil_million_sm3"] for row in production["series"]["JOHAN SVERDRUP"] if row["oil_million_sm3"] is not None)
troll_gas_total = sum(row["gas_billion_sm3"] for row in production["series"]["TROLL"] if row["gas_billion_sm3"] is not None)
display(HTML(
    f"<p><b>2024 source totals:</b> Johan Sverdrup oil {johan_oil_total:.6f} million Sm³; "
    f"Troll gas {troll_gas_total:.6f} billion Sm³. The September gas lows in both "
    "series identify a month for a multi-year follow-up; this chart does not establish a cause.</p>"
))


### Build the same production comparison inside Visual

The earlier Python figure is an independent check. This cell validates a parameterized API query and prints an equivalent bounded literal query for the current text-only Query editor. Paste the printed query into **Query**, leave **Show in graph** unchecked, and select **Run**, open **Data → Query results → Visualize result**, choose **Line**, map `month` to x, `monthly_oil_million_sm3` to y, and `field` to series. To show a monthly-average calendar-day rate, enable that transform, confirm the monthly contract, and use scale `1000000`. Suggested title: **Selected NCS fields — 2024 monthly-average oil rate**. The source table remains available beside the chart, and SVG/PNG export carries the query provenance.


In [ ]:
PRODUCTION_CHART_QUERY = """MATCH (p:ProductionProfile)-[:OF_FIELD]->(f:Field)
WHERE f.title IN $fields
UNWIND ts_series(p.prd_oil_net, '2024', '2024') AS observation
RETURN f.title AS field, observation.time AS month,
       observation.value AS monthly_oil_million_sm3
ORDER BY field, month
LIMIT 100"""
production_chart_rows = table_rows(
    PRODUCTION_CHART_QUERY, {"fields": FIELDS}, label="flat monthly oil rows for Visual"
)
production_counts = {
    field: sum(row["field"] == field for row in production_chart_rows)
    for field in FIELDS
}
assert set(production_counts) == set(FIELDS)
assert all(1 <= count <= 12 for count in production_counts.values())
assert len(production_chart_rows) <= 12 * len(FIELDS)
PRODUCTION_VISUAL_QUERY = PRODUCTION_CHART_QUERY.replace("$fields", json.dumps(FIELDS))
display_table(production_chart_rows[:6], "First six validated monthly oil rows")
display(HTML(
    "<p><b>Selected-field coverage:</b> "
    + html_lib.escape(", ".join(f"{field}: {count} months" for field, count in production_counts.items()))
    + "</p><p>Paste this bounded query into <b>Query</b>, leave <b>Show in graph</b> unchecked, select <b>Run</b>, then "
      "<b>Visualize result</b>. Choose <b>Line</b>, review the mapping, and select "
      "<b>Build chart</b>:</p>"
    + f'<pre style="white-space:pre-wrap">{html_lib.escape(PRODUCTION_VISUAL_QUERY)}</pre>'
))


## NJU-1 supported-play oil creaming curve

The public graph links a discovery to every compatible play supported by an authoritative published discovery/play example or by the designated discovery well’s age evidence inside a play polygon. Field membership alone does not assign a discovery to a play. A discovery may therefore have multiple `Discovery -[:IN_PLAY]-> Play` relationships. Each relationship records its source and geological evidence. The selected-play curve counts each discovery ID once even when duplicate relationship rows exist. Curves for different plays can contain the same discovery, so their subtotals overlap and must not be added together. Review the coverage tables before interpreting the curve.

The cell retains every matched discovery in a coverage table. The curve's chronology uses the designated discovery well's exact completion date; the reported discovery year remains in the coverage table for comparison. Discoveries without a completion date remain explicit in coverage and are not assigned an invented date. Contributions sharing a completion date are aggregated into one event. The primary curve uses reported and conservatively generated discovery oil resources. For each discovery it selects the newest usable `DiscoveryVolume` observation on a current oil-estimate basis. Reported resource-class rows remain distinct observations and conservative single-discovery field estimates identify their generated basis. Inclusion-window deltas use a different dated basis and are excluded from this current subtotal. Missing, conflicted, and unresolved estimates remain gaps and are reported. Public field estimates are shown only as separately dated context; they are not apportioned to discoveries or added to the curve. The result is a **known discovery-oil subtotal**, not a play oil-volume estimate. These are current estimates plotted against discovery chronology, not estimates known at each historical date. Values are million Sm³ recoverable oil; gas, NGL, and condensate are excluded. The approximate Troll allocation assigns all generated Troll oil to West and an explicit zero to East. That zero is a generated estimate, not missing data.

The Python cell computes cumulative rows because this Cypher surface has no window sum and prints a bounded literal query for Visual. Paste it into the public workspace, leave **Show in graph** unchecked, select **Run**, then **Visualize result**. The suggested line mapping uses `discovery_well_completion_date` as x, `cumulative_mill_sm3_oil` as y, and `series` as colour. Missing assets retain positions on the x axis as explicit gaps; the known-only subtotal resumes after each gap without adding an invented value.

The coverage also shows Gjøa Nord and Duva as multi-play examples. The generated sourced-estimate fallback uses the newest catalogued whole-discovery drilling-report estimate and never sums estimates from separate wells. Its current bounded catalog contains four verified reports; it does not imply that every discovery announcement has been ingested. The examples include a published 2022 Gjøa Nord range represented by a dated 2.8 million Sm³ oil-equivalent midpoint and a 2016 Duva discovery-report range represented by 7.65 million Sm³ oil equivalent. Those reports supply no oil component, so recoverable oil remains missing. Field-reserve increments are not used as a substitute. Only a usable explicitly sourced or generated oil component can enter the oil-only curve.

Set `SODIR_CREAMING_GRAPH` before starting the kernel only to evaluate a separately supplied compatible `.kgl` file; the public graph built by this notebook is the default.


In [ ]:
NJU1_MEMBERSHIP_QUERY = """MATCH (d:Discovery)-[membership:IN_PLAY]->(pl:Play {title: $play})
MATCH (d)-[:DISCOVERED_BY]->(well:Wellbore)
OPTIONAL MATCH (d)-[:IN_FIELD]->(f:Field)
OPTIONAL MATCH (volume:DiscoveryVolume)-[:OF_DISCOVERY]->(d)
RETURN DISTINCT d.id AS discovery_id, coalesce(d.dscName, d.title) AS discovery,
       d.dscDiscoveryYear AS reported_discovery_year,
       well.wlbCompletionDate AS discovery_well_completion_date,
       volume.recoverable_oil AS discovery_recoverable_mill_sm3_oil,
       volume.recoverable_oe AS discovery_recoverable_mill_sm3_oe,
       volume.estimate_date AS resource_snapshot,
       volume.resource_class AS discovery_resource_class,
       volume.generated AS volume_generated,
       volume.usable AS volume_usable,
       volume.method AS volume_method,
       volume.coverage AS volume_coverage,
       volume.basis AS volume_basis,
       volume.source_discovery_id AS volume_source_discovery_id,
       volume.source_record_identity AS volume_source_identity,
       volume.conflict AS volume_conflict,
       volume.unresolved_reason AS volume_unresolved_reason,
       f.id AS field_id, f.title AS field,
       membership.match_method AS match_method,
       membership.distance_m AS distance_m,
       membership.age_compatibility AS age_compatibility,
       membership.matched_ages AS matched_ages,
       membership.matched_hc_slots AS matched_hc_slots,
       membership.wellbore_ages AS wellbore_ages,
       membership.play_ages AS play_ages,
       membership.distance_method AS distance_method,
       membership.wlbNpdidWellbore AS source_wellbore_id,
       membership.source AS membership_source,
       membership.inferred AS inferred,
       membership.ambiguous AS ambiguous,
       membership.candidate_tie_count AS candidate_tie_count
ORDER BY discovery_well_completion_date, discovery_id
LIMIT 100"""

NJU1_PLAY_COVERAGE_QUERY = """MATCH (d:Discovery)-[:IN_PLAY]->(target:Play {title: $play})
MATCH (d)-[candidate:CANDIDATE_PLAY]->(candidate_play:Play)
RETURN candidate.rejection_reason AS rejection_reason,
       candidate.match_method AS candidate_match_method,
       candidate.age_compatibility AS candidate_age_compatibility,
       count(*) AS rejected_candidate_count
ORDER BY rejection_reason, candidate_match_method, candidate_age_compatibility
LIMIT 100"""

DISCOVERY_ASSIGNMENT_COVERAGE_QUERY = """MATCH (d:Discovery)
OPTIONAL MATCH (d)-[:IN_PLAY]->(assigned:Play)
RETURN count(DISTINCT d) AS total_discoveries,
       count(DISTINCT CASE WHEN assigned IS NOT NULL THEN d ELSE null END) AS assigned_discoveries
LIMIT 1"""

DUVA_REPORT_QUERY = """MATCH (d:Discovery)-[:DISCOVERED_BY]->(well:Wellbore)
WHERE d.dscName = $discovery AND well.id = $wellbore_id
RETURN coalesce(d.dscName, d.title) AS discovery,
       well.wlbWellboreName AS wellbore,
       well.wlbCompletionDate AS completion_date,
       well.wlbPressReleaseUrl AS source_url
LIMIT 1"""

DISCOVERY_PLAY_VOLUME_CONTEXT_QUERY = """MATCH (d:Discovery)-[membership:IN_PLAY]->(play:Play)
WHERE d.dscName = $discovery
OPTIONAL MATCH (volume:DiscoveryVolume)-[:OF_DISCOVERY]->(d)
RETURN DISTINCT d.id AS discovery_id,
       coalesce(d.dscName, d.title) AS discovery,
       play.title AS play,
       membership.source AS membership_source,
       membership.match_method AS membership_method,
       volume.estimate_date AS estimate_date,
       volume.recoverable_oe AS recoverable_mill_sm3_oe,
       volume.recoverable_oil AS recoverable_mill_sm3_oil,
       volume.method AS volume_method,
       volume.basis AS volume_basis,
       volume.generated AS generated,
       volume.usable AS usable,
       volume.unresolved_reason AS unresolved_reason,
       volume.source_record_identity AS source_record_identity,
       volume.source_record_json AS source_record_json,
       volume.generation_parameters_json AS generation_parameters_json
ORDER BY play, estimate_date, volume_method
LIMIT 50"""

NJU1_FIELD_CONTEXT_QUERY = """MATCH (d:Discovery)-[:IN_PLAY]->(pl:Play {title: $play})
MATCH (d)-[:IN_FIELD]->(field:Field)
WITH DISTINCT field
MATCH (reserve:FieldReserves)-[:OF_FIELD]->(field)
WITH field, max(reserve.title) AS latest_snapshot
MATCH (latest:FieldReserves)-[:OF_FIELD]->(field)
WHERE latest.title = latest_snapshot
RETURN field.id AS field_id, field.title AS field,
       latest.title AS resource_snapshot,
       latest.fldRecoverableOil AS field_original_recoverable_mill_sm3_oil
ORDER BY field
LIMIT 100"""

_creaming_view = _sodir_view
if CREAMING_GRAPH_PATH is not None:
    if not CREAMING_GRAPH_PATH.is_file():
        raise FileNotFoundError(f"SODIR_CREAMING_GRAPH does not exist: {CREAMING_GRAPH_PATH}")
    previous_creaming_view = globals().get("_external_creaming_view")
    if previous_creaming_view is not None and not previous_creaming_view.closed:
        previous_creaming_view.close()
    _external_creaming_view = kv.show(str(CREAMING_GRAPH_PATH), open_browser=False,
                                      max_load_mb=1024, name="SODIR NJU-1 creaming", height=680)
    _creaming_view = _external_creaming_view

membership = table_rows(NJU1_MEMBERSHIP_QUERY, {"play": "nju-1"},
                        label="supported NJU-1 discovery membership", view=_creaming_view)
if not membership:
    raise RuntimeError("the graph has no supported NJU-1 discovery membership")
play_coverage = table_rows(NJU1_PLAY_COVERAGE_QUERY, {"play": "nju-1"},
                           label="NJU-1 discovery play-candidate coverage", view=_creaming_view)
assignment_coverage = table_rows(DISCOVERY_ASSIGNMENT_COVERAGE_QUERY,
                                 label="all-discovery assignment coverage", view=_creaming_view)[0]
field_context_rows = table_rows(NJU1_FIELD_CONTEXT_QUERY, {"play": "nju-1"},
                                label="separate NJU-1 field-resource context", view=_creaming_view)
gjoa_nord_context = table_rows(
    DISCOVERY_PLAY_VOLUME_CONTEXT_QUERY, {"discovery": "35/9-3 (Gjøa Nord)"},
    label="Gjøa Nord multi-play and volume context", view=_creaming_view,
)
duva_context = table_rows(
    DISCOVERY_PLAY_VOLUME_CONTEXT_QUERY, {"discovery": "36/7-4 Duva"},
    label="Duva multi-play and volume context", view=_creaming_view,
)
duva_report = table_rows(
    DUVA_REPORT_QUERY, {"discovery": "36/7-4 Duva", "wellbore_id": 7988},
    label="Duva discovery-report source URL", view=_creaming_view,
)
assert len(duva_report) == 1 and str(duva_report[0]["source_url"]).startswith("https://")
assert {row["play"] for row in gjoa_nord_context} == {"nkl-2", "nku-5"}
gjoa_midpoints = [row for row in gjoa_nord_context
                  if row["volume_method"] == "published_resource_range_midpoint"]
assert gjoa_midpoints
assert all(row["estimate_date"] == "2022-05-12"
           and math.isclose(float(row["recoverable_mill_sm3_oe"]), 2.8)
           and row["recoverable_mill_sm3_oil"] is None
           and row["volume_basis"] == "newest_applicable_published_discovery_estimate"
           for row in gjoa_midpoints)
assert {row["play"] for row in duva_context} == {"nkl-2", "nku-5"}
duva_midpoints = [row for row in duva_context
                  if row["volume_method"] == "published_resource_range_midpoint"]
assert duva_midpoints
assert all(row["estimate_date"] == "2016-09-16"
           and math.isclose(float(row["recoverable_mill_sm3_oe"]), 7.65)
           and row["recoverable_mill_sm3_oil"] is None
           and row["volume_basis"] == "newest_applicable_published_discovery_estimate"
           for row in duva_midpoints)

def prepare_field_context(rows):
    latest = {}
    for row in rows:
        previous = latest.get(row["field_id"])
        if previous is None or str(row["resource_snapshot"]) > str(previous["resource_snapshot"]):
            latest[row["field_id"]] = row
    return sorted(latest.values(), key=lambda row: row["field"])

field_context = prepare_field_context(field_context_rows)

def prepare_discovery_assets(rows, *,
                             value_field="discovery_recoverable_mill_sm3_oil",
                             accepted_bases=None):
    """Select one component's newest usable observations without allocating field shares."""
    current_bases = (set(accepted_bases) if accepted_bases is not None else
                     {"discovery_reserves", "latest_original_recoverable_field_snapshot",
                      "latest_original_recoverable"})
    grouped = {}
    for row in rows:
        item = grouped.setdefault(row["discovery_id"], {
            "completion_date": row["discovery_well_completion_date"],
            "reported_year": row["reported_discovery_year"],
            "asset": row["discovery"], "volumes": [],
        })
        if row["volume_method"] is not None:
            item["volumes"].append({
                "snapshot": row["resource_snapshot"], "class": row["discovery_resource_class"],
                "value": row[value_field],
                "generated": row["volume_generated"], "usable": row["volume_usable"],
                "method": row["volume_method"], "coverage": row["volume_coverage"],
                "basis": row["volume_basis"], "source": row["volume_source_discovery_id"],
                "identity": row["volume_source_identity"], "conflict": row["volume_conflict"],
                "unresolved_reason": row["volume_unresolved_reason"],
            })
    assets = []
    for discovery_id, item in grouped.items():
        volumes = item.pop("volumes")
        eligible = []
        for row in volumes:
            try:
                value_is_finite = (row["value"] is not None
                                   and math.isfinite(float(row["value"])))
            except (TypeError, ValueError):
                value_is_finite = False
            if (row["usable"] is True and row["conflict"] is not True
                    and row["basis"] in current_bases and value_is_finite):
                eligible.append(row)
        snapshots = [row["snapshot"] for row in eligible if row["snapshot"] is not None]
        current = max(snapshots, key=lambda value: str(value)) if snapshots else None
        current_rows = [row for row in eligible if row["snapshot"] == current]
        identities = set()
        values = []
        for row in current_rows:
            identity = (row["method"], row["source"], row["identity"], row["class"])
            if identity in identities:
                continue
            identities.add(identity)
            if row["value"] is not None and math.isfinite(float(row["value"])):
                values.append(float(row["value"]))
        item.update({
            "value": sum(values) if values else None, "resource_snapshot": current,
            "volume_methods": sorted({row["method"] for row in current_rows}),
            "generated": any(row["generated"] is True for row in current_rows),
            "excluded_observations": len(volumes) - len(current_rows),
        })
        assets.append(item)
    return sorted(assets, key=lambda row: (
        row["completion_date"] is None, str(row["completion_date"] or ""),
        row["reported_year"] is None, row["reported_year"] or 0, row["asset"],
    ))

assets = prepare_discovery_assets(membership)
nkl2_membership = table_rows(
    NJU1_MEMBERSHIP_QUERY, {"play": "nkl-2"},
    label="supported NKL-2 discovery membership", view=_creaming_view,
)
nkl2_oil_assets = prepare_discovery_assets(nkl2_membership)
nkl2_oe_assets = prepare_discovery_assets(
    nkl2_membership,
    value_field="discovery_recoverable_mill_sm3_oe",
    accepted_bases={"discovery_reserves", "latest_original_recoverable_field_snapshot",
                    "latest_original_recoverable", "newest_applicable_published_discovery_estimate"},
)
def finite_resource(row):
    try:
        return row["value"] is not None and math.isfinite(float(row["value"]))
    except (TypeError, ValueError):
        return False
missing = [row for row in assets if not finite_resource(row)]
known_total = 0.0
creaming_points = []
missing_completion_dates = [row for row in assets if row["completion_date"] is None]
dated_assets = [row for row in assets if row["completion_date"] is not None]
for completion_date in sorted({row["completion_date"] for row in dated_assets}, key=str):
    events = [row for row in dated_assets if row["completion_date"] == completion_date]
    known_total += sum(float(row["value"]) for row in events if finite_resource(row))
    creaming_points.append({
        "discovery_well_completion_date": str(completion_date),
        "series": "reported and generated discovery-oil subtotal",
        "cumulative_mill_sm3_oil": (None if any(not finite_resource(row) for row in events)
                                    else known_total),
    })

CREAMING_CHART_QUERY = """UNWIND $points AS point
RETURN point.discovery_well_completion_date AS discovery_well_completion_date,
       point.cumulative_mill_sm3_oil AS cumulative_mill_sm3_oil,
       point.series AS series
ORDER BY discovery_well_completion_date, series
LIMIT 300"""
assert len(creaming_points) <= 300
chart_rows = table_rows(CREAMING_CHART_QUERY, {"points": creaming_points},
                        label="NJU-1 known-oil subtotal rows", view=_creaming_view)
def point_literal(point):
    value = point["cumulative_mill_sm3_oil"]
    y = "null" if value is None else repr(value)
    series = json.dumps(point["series"])
    completion_date = json.dumps(point["discovery_well_completion_date"])
    return ("{discovery_well_completion_date:" + completion_date
            + ",cumulative_mill_sm3_oil:" + y
            + ",series:" + series + "}")
literal_points = "[" + ",".join(map(point_literal, creaming_points)) + "]"
CREAMING_VISUAL_QUERY = CREAMING_CHART_QUERY.replace("$points", literal_points)
display(HTML(
    f'<p><a href="{_creaming_view.url}" target="_blank"><b>Open the NJU-1 workspace</b></a></p>'
    f'<p>Paste this bounded query into <b>Query</b>, leave <b>Show in graph</b> unchecked, select <b>Run</b>, then <b>Visualize result</b>. Review the suggested line mapping and select <b>Build chart</b>:</p>'
    f'<pre style="white-space:pre-wrap">{html_lib.escape(CREAMING_VISUAL_QUERY)}</pre>'
))
assert len(chart_rows) == len(creaming_points)
display_table(membership, "Supported NJU-1 membership and current discovery-oil coverage")
display_table(nkl2_membership, "Supported NKL-2 membership and oil/OE coverage")
nkl2_oil_known = [row for row in nkl2_oil_assets if finite_resource(row)]
nkl2_oe_known = [row for row in nkl2_oe_assets if finite_resource(row)]
display(HTML(
    f"<p><b>NKL-2 coverage:</b> oil {len(nkl2_oil_known)} of {len(nkl2_oil_assets)} "
    f"discoveries, subtotal {sum(row['value'] for row in nkl2_oil_known):.3f} million Sm³; "
    f"oil equivalent {len(nkl2_oe_known)} of {len(nkl2_oe_assets)}, subtotal "
    f"{sum(row['value'] for row in nkl2_oe_known):.3f} million Sm³. "
    "Missing component values remain missing. These overlapping play results must not be "
    "added to another play's subtotal.</p>"
))
display_table(play_coverage, "Rejected play candidates for NJU-1-assigned discoveries")
display_table(field_context, "Dated field-resource context (not allocated or added to the curve)")
display_table(gjoa_nord_context, "Gjøa Nord: both supported plays and dated OE context")
display_table(duva_context, "Duva: both supported plays and dated OE context")
display_table(duva_report, "Duva source well and discovery-report URL")
methods = sorted({row["match_method"] for row in membership})
age_coverage = sorted({row["age_compatibility"] for row in membership
                       if row["age_compatibility"] is not None})
display(HTML(
    f"<p><b>Coverage:</b> {len(assets)} NJU-1-assigned discoveries; "
    f"{len(assets) - len(missing)} discoveries have an eligible current estimate; "
    f"{len(missing)} lack an eligible current-basis volume observation; "
    f"{len(missing_completion_dates)} lack a designated-well completion date. "
    f"{assignment_coverage['assigned_discoveries']} of "
    f"{assignment_coverage['total_discoveries']} public discoveries have at least one supported play; "
    f"{sum(row['rejected_candidate_count'] for row in play_coverage)} rejected candidate edges "
    "are retained for this NJU-1 subset. "
    f"Methods: {html_lib.escape(', '.join(methods))}; age compatibility: "
    f"{html_lib.escape(', '.join(age_coverage))}. "
    "Memberships come from published discovery examples or compatible well age/polygon evidence; "
    "the subtotal is not a complete play oil-volume estimate.</p>"
))


## 5. Preserve a reviewable well-level handoff

Return to the specific 16/2-6 evidence assembled above: ordered formation tops, three cores, and one DST record. This is a reviewable source neighborhood rather than an unordered sample across hundreds of matching field patterns. Save it under a notebook-specific name, then preview and download its exact visible GraphML membership.

In [ ]:
handoff_view = show_graph(COMBINED_EVIDENCE_QUERY, {"well": "16/2-6"},
                          label="16/2-6 depth-context handoff", kernel="radial",
                          color_by="type")
current = request("GET", "/api/view-state")
saved = request("POST", "/api/views/save", {
    "name": "SODIR notebook — 16/2-6 depth context", "replace": True,
    "expected": current["stamp"],
})
current = request("GET", "/api/view-state")
preview_request = {
    "scope": "visible", "format": "graphml", "expected": current["stamp"],
    "subset_revision": current["subset_revision"],
}
preview = request("POST", "/api/export/preview", preview_request)
payload, headers = request("POST", "/api/export/download",
                           {**preview_request, "preview_digest": preview["preview_digest"]}, raw=True)
export_path = EXPORT_DIR / "well-16-2-6-depth-context.graphml"
export_path.write_bytes(payload)
print(f"Saved view and exported {preview['nodes']} nodes / "
      f"{preview['edges']} relations to {export_path.name}")
display(HTML(f'<a href="{_sodir_view.url}" target="_blank">Continue in the live workspace</a>'))


## Optional cleanup

**Run All stops above and leaves the workspace running.** Execute the first cell below only when you are finished. The second cell deletes notebook-generated exports and figures only after you change its guard. The downloaded SODIR cache and portable graph remain available for the next run.

In [ ]:
# Optional: stop only the viewer owned by this notebook.
# _sodir_view.close()

# if "_creaming_view" in globals() and not _creaming_view.closed:
#     _creaming_view.close()


In [ ]:
# Optional: remove small notebook-generated artifacts, never the source cache or graph.
REMOVE_NOTEBOOK_OUTPUTS = False
if REMOVE_NOTEBOOK_OUTPUTS:
    import shutil
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
    shutil.rmtree(FIGURE_DIR, ignore_errors=True)
    print("Removed notebook-owned exports and figures.")
